# InnoBERT: installation and user manual

This notebook leads a new user from installation through model loading, term classification, noun-chunk extraction, sentence and paragraph processing, DataFrame batches, threshold adjustment, diagnostics, and saving results.

The same examples run on CPU or GPU. `device="auto"` selects the best available device. The innovation-type framework is informed by the Oslo Manual 2018; InnoBERT additionally reports sustainability, AI, and an uncategorized outcome.

## 1. Choose one installation route

### Route A — New local Conda environment

1. Install [Anaconda or Miniconda](https://www.anaconda.com/docs/getting-started/main).
2. Open **Anaconda Prompt** on Windows or a terminal on macOS/Linux.
3. Run:

```bash
conda create -n innobert python=3.11 -y
conda activate innobert
python -m pip install --upgrade pip
python -m pip install "innobert[noun-chunks,notebook] @ git+https://github.com/mustafahci/InnoBERT.git@main"
python -m spacy download en_core_web_lg
python -m ipykernel install --user --name innobert --display-name "Python (InnoBERT)"
jupyter lab
```

Open this notebook and select **Python (InnoBERT)** as its kernel.

### Route B — Existing Python/IPython environment

Activate the intended environment and run:

```bash
python -m pip install "innobert[noun-chunks,notebook] @ git+https://github.com/mustafahci/InnoBERT.git@main"
python -m spacy download en_core_web_lg
```

### Route C — Google Colab

Open the notebook in Colab and run the next installation cell. Select a GPU runtime before loading the model if GPU execution is desired. Colab runtimes are temporary, so installation and authentication must be repeated after a runtime reset.

In [ ]:
# Run this cell only in Google Colab or an environment where InnoBERT is not installed.
# %pip install "innobert[noun-chunks] @ git+https://github.com/mustafahci/InnoBERT.git@main"
# !python -m spacy download en_core_web_lg

## 2. Verify the active Python environment

The Python path should belong to the environment selected as the notebook kernel.

In [ ]:
import sys
import innobert

print("Python:", sys.executable)
print("InnoBERT:", innobert.__version__)

## 3. Authenticate with Hugging Face

Authentication is needed while the model repository is restricted. The secure prompt stores the token outside the notebook. Never paste a token into a notebook cell.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

Terminal and script users can instead run `hf auth login` once in the activated environment.

## 4. Load InnoBERT

`device="auto"` selects CUDA, Apple MPS, or CPU. The spaCy model is loaded only when noun-chunk processing is requested.

In [ ]:
import torch
from innobert import InnoBERT

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

classifier = InnoBERT.from_pretrained(
    "mustafahci/InnoBERT",
    device="auto",
    spacy_model="en_core_web_lg",
)
print("Model loaded successfully on", classifier.device)

## 5. Supply your own terms

Each term can have a different industry and year. A scalar is broadcast; a list must align exactly with the terms. Term and noun-chunk modes use the validated industry–year prompt and uncategorized gatekeeper by default.

In [ ]:
terms = [
    "cloud-based analytics platform",
    "automated inventory replenishment system",
    "employee collaboration network",
]

term_results = classifier.predict(
    terms,
    industry=["Software", "Retail", "Manufacturing"],
    year=[2024, 2023, 2022],
    unit="term",
    progress=False,
)
term_results

Illustrative CPU output from the verified walkthrough:

| processed_text | predicted_subcategory_labels | predicted_subcategory_probabilities | main_categories | highest_probability_label | highest_probability |
| --- | --- | --- | --- | --- | ---: |
| cloud-based analytics platform | [product] | [0.931235] | [product] | product | 0.931235 |
| automated inventory replenishment system | [process] | [0.976357] | [business_process] | process | 0.976357 |
| employee collaboration network | [organizational] | [0.942569] | [business_process] | organizational | 0.942569 |

Probabilities remain aligned with their subcategory labels. Small numerical differences can occur across hardware and library versions.

## 6. Use one raw passage in all three processing modes

The next passage is intentionally long enough to contain several types of innovation and non-innovation content.

In [ ]:
long_text = """Example Corporation expanded its cloud-based subscription platform for small-business customers and introduced personalized product recommendations through mobile account management. It deployed an artificial-intelligence forecasting tool, real-time analytics, and an automated inventory replenishment system.

The company created a cross-functional product team and an employee collaboration network. A data-driven order-routing process improved warehouse allocation, distribution operations, purchasing, and delivery scheduling. These initiatives also reduced short-term cash requirements.

It adopted an energy-efficient manufacturing process and recyclable packaging materials to reduce operating emissions and waste during the year."""

### 6.1 Noun-chunk processing

The source is parsed by spaCy, candidate terms are filtered, and each retained term is classified separately. Industry and year are required by default.

In [ ]:
noun_results = classifier.predict(
    long_text,
    industry="Business Services",
    year=2024,
    filer_name="Example Corporation",
    unit="noun_chunk",
    progress="auto",
)
noun_results

Selected noun-chunk output from the verified CPU walkthrough:

| processed_text | predicted_subcategory_labels | main_categories | highest_probability_label | highest_probability |
| --- | --- | --- | --- | ---: |
| artificial-intelligence forecasting tool | [AI] | [AI] | AI | 0.923239 |
| automated inventory replenishment system | [process] | [business_process] | process | 0.974152 |
| cloud-based subscription platform | [product, business_model] | [product, business_process] | business_model | 0.941937 |
| cross-functional product team | [organizational] | [business_process] | organizational | 0.898596 |
| employee collaboration network | [organizational] | [business_process] | organizational | 0.920319 |
| energy-efficient manufacturing process | [process, sustainability] | [business_process, sustainability] | process | 0.899751 |
| personalized product recommendation | [product, AI] | [product, AI] | AI | 0.839709 |
| recyclable packaging material | [sustainability] | [sustainability] | sustainability | 0.965128 |

Generic phrases can be assigned `uncategorized`. The full result also contains the probability associated with every selected subcategory.

### 6.2 Sentence processing

Sentence mode returns one row per detected sentence. It uses the raw sentence and fallback uncategorized rule by default.

In [ ]:
sentence_results = classifier.predict(
    long_text,
    unit="sentence",
    progress=False,
)
sentence_results

Representative verified sentence-level CPU output:

| processed_text | predicted_subcategory_labels | main_categories | highest_probability_label | highest_probability |
| --- | --- | --- | --- | ---: |
| We launched a cloud-based platform ... | [product, business_model] | [product, business_process] | product | 0.758347 |
| We automated inventory replenishment ... | [process] | [business_process] | process | 0.948941 |
| We introduced cross-functional teams ... | [organizational] | [business_process] | organizational | 0.803438 |

### 6.3 Paragraph processing

Blank lines define paragraphs. A paragraph longer than the model limit is divided into overlapping windows; category-wise maximum probabilities are returned.

In [ ]:
paragraph_results = classifier.predict(
    long_text,
    unit="paragraph",
    progress=False,
)
paragraph_results

Representative verified paragraph-level CPU output:

| processed_text | predicted_subcategory_labels | main_categories | highest_probability_label | highest_probability |
| --- | --- | --- | --- | ---: |
| We launched a cloud-based platform ... | [product] | [product] | product | 0.692491 |
| We automated inventory replenishment ... | [process] | [business_process] | process | 0.947389 |

## 7. Inspect full probabilities and processing diagnostics

Compact output is the default. Full output adds all eight probabilities, source lineage, `main_categories`, token/window counts, long-input actions, and device information.

In [ ]:
full_sentence_results = classifier.predict(
    long_text,
    unit="sentence",
    output="full",
    progress=False,
)

probability_columns = [column for column in full_sentence_results if column.startswith("prob_")]
full_sentence_results[
    ["unit_id", "processed_text", "predicted_subcategory_labels", "main_categories", *probability_columns]
].head()

## 8. Process a DataFrame of long passages

The output remains at the extracted-unit level. `source_id_cols` creates a traceable source key, while `metadata_cols` copies identifiers to every resulting row.

In [ ]:
import pandas as pd

documents = pd.DataFrame({
    "gvkey": ["001234", "001234"],
    "fyear": [2023, 2024],
    "Industry": ["Business Services", "Business Services"],
    "long_passage": [long_text, long_text],
})

dataframe_results = classifier.predict(
    documents,
    unit="noun_chunk",
    text_col="long_passage",
    industry_col="Industry",
    year_col="fyear",
    source_id_cols=["gvkey", "fyear"],
    metadata_cols=["gvkey", "fyear"],
    progress="auto",
)
dataframe_results.head(10)

The package deliberately does not create firm-year counts, indicators, or scores. Users can aggregate the unit-level output according to their research design.

## 9. Adjust probability thresholds

Unspecified categories retain the published defaults. Use full output to evaluate probabilities before adopting corpus-specific thresholds.

In [ ]:
adjusted_results = classifier.predict(
    long_text,
    unit="sentence",
    thresholds={"product": 0.70, "AI": 0.40},
    output="full",
    progress=False,
)

adjusted_results[
    ["unit_id", "prob_inno_product", "prob_inno_AI", "predicted_subcategory_labels"]
]

Published defaults:

```python
product = 0.65
process = 0.45
organizational = 0.55
marketing = 0.55
business_model = 0.45
sustainability = 0.50
AI = 0.50
uncategorized = 0.25
```

## 10. Long documents and complete 10-K filings

A 100-page filing must not be treated as one `term`. Select a unit:

- `noun_chunk`: parse the filing and classify extracted short terms;
- `sentence`: classify sentences separately;
- `paragraph`: classify paragraphs and window unusually long paragraphs.

Long term, noun-chunk, and sentence units raise an error by default instead of being silently truncated. Paragraph mode windows long paragraphs automatically. If paragraph breaks were lost and a whole filing appears as one paragraph, InnoBERT issues a warning.

For the research measure, extract the intended filing section before classification. A complete 10-K includes risk factors, MD&A, notes, and other sections that can change the construct. If complete filings are required, preserve `section_id` and process sections as separate source rows.

## 11. Progress and run summary

`progress="auto"` displays progress in interactive notebooks and terminals, while remaining silent in redirected or automated jobs. Use `True` to force it or `False` for clean logs.

In [ ]:
classifier.last_run_summary

## 12. Save results

CSV is widely compatible but stores list-valued columns as text. Parquet preserves list-like values more naturally.

In [ ]:
dataframe_results.to_csv("innobert_noun_chunks.csv", index=False)
dataframe_results.to_parquet("innobert_noun_chunks.parquet", index=False)

sentence_results.to_csv("innobert_sentences.csv", index=False)
paragraph_results.to_csv("innobert_paragraphs.csv", index=False)

print("Saved output files.")

## 13. Informative errors

This example intentionally provides two terms but only one industry. The error identifies the alignment problem.

In [ ]:
try:
    classifier.predict(
        ["first term", "second term"],
        industry=["Software"],
        year=[2024, 2023],
        unit="term",
        progress=False,
    )
except (TypeError, ValueError, RuntimeError, ImportError, OSError) as error:
    print(type(error).__name__ + ":", error)

Other explicit checks cover missing context, duplicate source identifiers, invalid years and thresholds, unavailable devices, missing spaCy resources, unsupported modes, and unsafe long inputs.

## 14. Use InnoBERT without a notebook

Save ordinary Python code as `run_innobert.py` and execute `python run_innobert.py` from the activated environment. The API is identical. Set `progress=False` for scheduled jobs and log files.

## Citation

Ahci, Mustafa and Joos, Philip, **Beyond Invention: The Composition and Economic Relevance of Innovation-Related Capabilities** (Updated September 1, 2026). Available at SSRN: [https://ssrn.com/abstract=4797745](https://ssrn.com/abstract=4797745) or [http://dx.doi.org/10.2139/ssrn.4797745](http://dx.doi.org/10.2139/ssrn.4797745).

OECD/Eurostat (2018), *Oslo Manual 2018: Guidelines for Collecting, Reporting and Using Data on Innovation*, 4th Edition, The Measurement of Scientific, Technological and Innovation Activities, OECD Publishing, Paris/Eurostat, Luxembourg. [https://doi.org/10.1787/9789264304604-en](https://doi.org/10.1787/9789264304604-en).